# Training-scaling with Nparams at fixed Dloc=3: bias vs. effective dimension

This notebook reproduces the paper's Nparams scaling results at fixed local per-parameter basis dimension Dloc=3. It compares the training-performance gap `Delta_{f-c}MSE_min = MSE_full_min - MSE_cut_min` and the normalized effective dimension (ED, Eq. 12) of a "full" (high-ED) partial Fourier model against a "cutoff" (lower-ED, decay-truncated) counterpart, as a function of the number of trainable parameters Nparams (M). Results are compared for the biased (target function exactly reproducible by both models) and unbiased (generic/unrelated target function) data-generating regimes produced by the companion scripts `train_and_FIM_biased.py`/`train_and_FIM_UNbiased.py`, across several numbers of training points per feature (Ntrain). The notebook loads the pickled per-model-draw results, aggregates them across random model/data draws, and reproduces the paper's plots of `Delta_{f-c}MSE_min` against Nparams.

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import re
import pickle

import pennylane.numpy as np



import matplotlib
import matplotlib.pyplot as plt



# Current path for importing custom functions
path_base = '/home/b/b309245/cloud_cover_qml_parametrization/DYAMOND_cellbased_implementations/'
sys.path.insert(0, path_base + 'qml_useful_functions')

import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Sets the experiment identifiers (results folder, biased/unbiased data-gen tags, learning rate, batch size, `cutoff`, `decay_exp`, `bond_dim`, Dloc, the fixed `max_frequency`, and the currently selected Ntrain = `no_training_data_per_feature`) and the scanned Nparams values (`no_params_vec`), matching those used when running `train_and_FIM_biased.py`/`train_and_FIM_UNbiased.py`, so the saved result filenames can be reconstructed and located.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ------------------------------------ Basic model specs ----------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

# Folder in which to load data from
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_training_with_Nparams_Dloc3/'

name_data_gen_fullbiased = 'biased_data_gen'
name_data_gen_unbiased = 'UNbiased_data_gen'

# Learning rate
learning_rate = 0.02
learning_rate_name = '0p02'

# Batch size for training
batch_size = 5
batch_size_name = str(batch_size)

### Cutoff index for correlations among Fourier components
cutoff = 3
cutoff_name = str(cutoff)

### No. of features (dimension of input vectors)
no_of_features = 1
name_no_features = str(no_of_features)

### Maximal Fourier frequency
max_frequency = 13
name_max_freq = str(max_frequency)

### No. basis states per parameter
dim_basis_single_param = 3  ### (1, cos(th), sin(th))
name_dim_basis_param = str(dim_basis_single_param)

# Used for setting the no. of training data
#no_training_data_per_feature = 10
#no_training_data_per_feature = 15
#no_training_data_per_feature = 20
#no_training_data_per_feature = 30
#no_training_data_per_feature = 40
no_training_data_per_feature = 60

# Bond dimension
bond_dim = 50
name_bond_dim = str(bond_dim)

### Cutoff index for correlations among Fourier components:
### the higher the cutoff, the more correlations the cut-off model will exhibit,
### and the higher the effective dimension will be.
### The SVs of the model's structure constants will start decaying after 'cutoff' values, i.e.,
### S_full = S0[0:]  for the full model
### S_cut[0:cutoff] = S0[0:cutoff]; S_cut[cutoff:] = decay_factor * S0[cutoff:]  for the cut model
cutoff = 3
cutoff_name = str(cutoff)

### Decay exponent for cutoff model: 
### S_cut[cutoff+i-1] = np.exp(- decay_exp * i) for i in [1, dim_basis_inputs-cutoff]
decay_exp = 3.0
decay_exp_name = '3p0'

### No. of parameters
no_params_vec = [5, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70, 80, 90, 100]

Aggregates the raw per-model-draw `dict_results...pkl` files for the currently configured Ntrain (`no_training_data_per_feature`) across all random model draws, separately for the biased and unbiased data-generating cases, into per-Nparams dictionaries of `Delta_{f-c}MSE_min` statistics, MSE training trajectories, and normalized FIM spectra/effective dimension.

In [ ]:
name_0 = 'dict_results'
name_1 = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
          '_Nparams')
name_2 = ('_DimBasisParam' + name_dim_basis_param + '_' + name_data_gen_fullbiased + 
          '_NtrainPerFeat' + str(no_training_data_per_feature) + '_batch' + batch_size_name + 
          '_lr' + learning_rate_name + '_cutoff' + cutoff_name + '_decayexp' + decay_exp_name + 
          '_modeldraw')
Ntrain_str = '_NtrainPerFeat' + str(no_training_data_per_feature)
filename0 = name_0 + name_1
listallfiles = [f for f in os.listdir(results_folder) 
                if (f.startswith(filename0) and (Ntrain_str in f) and (('_' + name_data_gen_fullbiased) in f))]
dict_fullbiased_all = dict()
for filename in listallfiles:
    res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + '(\S+).pkl'), filename)
    no_params_name = res[0][0]
    no_params = int(no_params_name)
    nm_name = res[0][1]
    nm = int(nm_name)
    dict_d = dict()
    dict_d['delta_minMSE_mean'] = []
    dict_d['delta_minMSE_std'] = []
    dict_d['delta_minMSE_min'] = []
    dict_d['delta_minMSE_max'] = []
    dict_d['MSE_full_mean'] = []
    dict_d['MSE_full_std'] = []
    dict_d['MSE_full_min'] = []
    dict_d['MSE_full_max'] = []
    dict_d['MSE_cut_mean'] = []
    dict_d['MSE_cut_std'] = []
    dict_d['MSE_cut_min'] = []
    dict_d['MSE_cut_max'] = []
    dict_d['mean_normFIM_spectra_full'] = []
    dict_d['std_normFIM_spectra_full'] = []
    dict_d['min_normFIM_spectra_full'] = []
    dict_d['max_normFIM_spectra_full'] = []
    dict_d['norm_eff_dim_full'] = []
    dict_d['mean_normFIM_spectra_cut'] = []
    dict_d['std_normFIM_spectra_cut'] = []
    dict_d['min_normFIM_spectra_cut'] = []
    dict_d['max_normFIM_spectra_cut'] = []
    dict_d['norm_eff_dim_cut'] = []
    dict_fullbiased_all[no_params] = dict_d
for filename in listallfiles:
    res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + '(\S+).pkl'), filename)
    no_params_name = res[0][0]
    no_params = int(no_params_name)
    nm_name = res[0][1]
    nm = int(nm_name)
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_model = pickle.load(f)
    dict_d = dict_fullbiased_all[no_params]
    dict_d['delta_minMSE_mean'].append(np.mean(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_std'].append(np.std(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_min'].append(np.min(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_max'].append(np.max(dict_model['delta_minMSE_full_m_cut']))
    dict_d['MSE_full_mean'].append(dict_model['MSE_full_mean'])
    dict_d['MSE_full_std'].append(dict_model['MSE_full_std'])
    dict_d['MSE_full_min'].append(dict_model['MSE_full_min'])
    dict_d['MSE_full_max'].append(dict_model['MSE_full_max'])
    dict_d['MSE_cut_mean'].append(dict_model['MSE_cut_mean'])
    dict_d['MSE_cut_std'].append(dict_model['MSE_cut_std'])
    dict_d['MSE_cut_min'].append(dict_model['MSE_cut_min'])
    dict_d['MSE_cut_max'].append(dict_model['MSE_cut_max'])
    dict_d['mean_normFIM_spectra_full'].append(dict_model['mean_normFIM_spectra_full'])
    dict_d['std_normFIM_spectra_full'].append(dict_model['std_normFIM_spectra_full'])
    dict_d['min_normFIM_spectra_full'].append(dict_model['min_normFIM_spectra_full'])
    dict_d['max_normFIM_spectra_full'].append(dict_model['max_normFIM_spectra_full'])
    dict_d['norm_eff_dim_full'].append(dict_model['norm_eff_dim_full'])
    dict_d['mean_normFIM_spectra_cut'].append(dict_model['mean_normFIM_spectra_cut'])
    dict_d['std_normFIM_spectra_cut'].append(dict_model['std_normFIM_spectra_cut'])
    dict_d['min_normFIM_spectra_cut'].append(dict_model['min_normFIM_spectra_cut'])
    dict_d['max_normFIM_spectra_cut'].append(dict_model['max_normFIM_spectra_cut'])
    dict_d['norm_eff_dim_cut'].append(dict_model['norm_eff_dim_cut'])
    dict_fullbiased_all[no_params] = dict_d


name_0 = 'dict_results'
name_1 = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
          '_Nparams')
name_2 = ('_DimBasisParam' + name_dim_basis_param + '_' + name_data_gen_unbiased + 
          '_NtrainPerFeat' + str(no_training_data_per_feature) + '_batch' + batch_size_name + 
          '_lr' + learning_rate_name + '_cutoff' + cutoff_name + '_decayexp' + decay_exp_name + 
          '_modeldraw')
Ntrain_str = '_NtrainPerFeat' + str(no_training_data_per_feature)
filename0 = name_0 + name_1
listallfiles = [f for f in os.listdir(results_folder) 
                if (f.startswith(filename0) and (Ntrain_str in f) and (('_' + name_data_gen_unbiased) in f))]
dict_unbiased_all = dict()
for filename in listallfiles:
    res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + '(\S+).pkl'), filename)
    no_params_name = res[0][0]
    no_params = int(no_params_name)
    nm_name = res[0][1]
    nm = int(nm_name)
    dict_d = dict()
    dict_d['delta_minMSE_mean'] = []
    dict_d['delta_minMSE_std'] = []
    dict_d['delta_minMSE_min'] = []
    dict_d['delta_minMSE_max'] = []
    dict_d['MSE_full_mean'] = []
    dict_d['MSE_full_std'] = []
    dict_d['MSE_full_min'] = []
    dict_d['MSE_full_max'] = []
    dict_d['MSE_cut_mean'] = []
    dict_d['MSE_cut_std'] = []
    dict_d['MSE_cut_min'] = []
    dict_d['MSE_cut_max'] = []
    dict_d['mean_normFIM_spectra_full'] = []
    dict_d['std_normFIM_spectra_full'] = []
    dict_d['min_normFIM_spectra_full'] = []
    dict_d['max_normFIM_spectra_full'] = []
    dict_d['norm_eff_dim_full'] = []
    dict_d['mean_normFIM_spectra_cut'] = []
    dict_d['std_normFIM_spectra_cut'] = []
    dict_d['min_normFIM_spectra_cut'] = []
    dict_d['max_normFIM_spectra_cut'] = []
    dict_d['norm_eff_dim_cut'] = []
    dict_unbiased_all[no_params] = dict_d
for filename in listallfiles:
    res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + '(\S+).pkl'), filename)
    no_params_name = res[0][0]
    no_params = int(no_params_name)
    nm_name = res[0][1]
    nm = int(nm_name)
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_model = pickle.load(f)
    dict_d = dict_unbiased_all[no_params]
    dict_d['delta_minMSE_mean'].append(np.mean(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_std'].append(np.std(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_min'].append(np.min(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_max'].append(np.max(dict_model['delta_minMSE_full_m_cut']))
    dict_d['MSE_full_mean'].append(dict_model['MSE_full_mean'])
    dict_d['MSE_full_std'].append(dict_model['MSE_full_std'])
    dict_d['MSE_full_min'].append(dict_model['MSE_full_min'])
    dict_d['MSE_full_max'].append(dict_model['MSE_full_max'])
    dict_d['MSE_cut_mean'].append(dict_model['MSE_cut_mean'])
    dict_d['MSE_cut_std'].append(dict_model['MSE_cut_std'])
    dict_d['MSE_cut_min'].append(dict_model['MSE_cut_min'])
    dict_d['MSE_cut_max'].append(dict_model['MSE_cut_max'])
    dict_d['mean_normFIM_spectra_full'].append(dict_model['mean_normFIM_spectra_full'])
    dict_d['std_normFIM_spectra_full'].append(dict_model['std_normFIM_spectra_full'])
    dict_d['min_normFIM_spectra_full'].append(dict_model['min_normFIM_spectra_full'])
    dict_d['max_normFIM_spectra_full'].append(dict_model['max_normFIM_spectra_full'])
    dict_d['norm_eff_dim_full'].append(dict_model['norm_eff_dim_full'])
    dict_d['mean_normFIM_spectra_cut'].append(dict_model['mean_normFIM_spectra_cut'])
    dict_d['std_normFIM_spectra_cut'].append(dict_model['std_normFIM_spectra_cut'])
    dict_d['min_normFIM_spectra_cut'].append(dict_model['min_normFIM_spectra_cut'])
    dict_d['max_normFIM_spectra_cut'].append(dict_model['max_normFIM_spectra_cut'])
    dict_d['norm_eff_dim_cut'].append(dict_model['norm_eff_dim_cut'])
    dict_unbiased_all[no_params] = dict_d


name_end_extr = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
                 '_DimBasisParam' + name_dim_basis_param + '_NtrainPerFeat' + str(no_training_data_per_feature) + 
                 '_batch' + batch_size_name + '_lr' + learning_rate_name + '_cutoff' + cutoff_name + 
                 '_decayexp' + decay_exp_name)
    
filename = 'extracted_results_FULLbiased' + name_end_extr + '.pkl'
path_file = os.path.join(results_folder, filename)
with open(path_file, 'wb') as f:
    pickle.dump(dict_fullbiased_all, f)

filename = 'extracted_results_UNbiased' + name_end_extr + '.pkl'
path_file = os.path.join(results_folder, filename)
with open(path_file, 'wb') as f:
    pickle.dump(dict_unbiased_all, f)

Loads the previously aggregated `extracted_results_FULLbiased`/`extracted_results_UNbiased` pickles for the currently configured Ntrain, providing the per-Nparams biased/unbiased `Delta_{f-c}MSE_min` and effective-dimension data used by the plots below.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ---------------------------------------- Load data --------------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

name_end_extr = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
                 '_DimBasisParam' + name_dim_basis_param + '_NtrainPerFeat' + str(no_training_data_per_feature) + 
                 '_batch' + batch_size_name + '_lr' + learning_rate_name + '_cutoff' + cutoff_name + 
                 '_decayexp' + decay_exp_name)
    
filename = 'extracted_results_FULLbiased' + name_end_extr + '.pkl'
path_file = os.path.join(results_folder, filename)
with open(path_file, 'rb') as f:
    dict_fullbiased_all = pickle.load(f)

filename = 'extracted_results_UNbiased' + name_end_extr + '.pkl'
path_file = os.path.join(results_folder, filename)
with open(path_file, 'rb') as f:
    dict_unbiased_all = pickle.load(f)

Plots the mean `Delta_{f-c}MSE_min` for the biased data-generating case (the unbiased curve is computed but plotted with a commented-out line) against Nparams (M) for the currently configured Ntrain, on a log MSE-difference scale, illustrating that the biased/low-ED cutoff model trains better than the full model when the target is biased.

In [ ]:
fs = 24
figsize = (10,4)

cmap = matplotlib.colormaps['cividis']

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

dMSE_biased = []
dMSE_unbiased = []
for no_params in no_params_vec:
    dict_nopars_biased = dict_fullbiased_all[no_params]
    deltaMSE_biased = np.asarray(dict_nopars_biased['delta_minMSE_mean'])
    dMSE_biased.append(np.mean(deltaMSE_biased))
    dict_nopars_unbiased = dict_unbiased_all[no_params]
    deltaMSE_unbiased = np.asarray(dict_nopars_unbiased['delta_minMSE_mean'])
    dMSE_unbiased.append(np.mean(deltaMSE_unbiased))

xx = np.asarray(no_params_vec)
yy_b = np.asarray(dMSE_biased)
yy_u = np.asarray(dMSE_unbiased)

ax.plot(xx, yy_b, 'o', color=cmap(0.0), markersize=7, 
        label=r'$\mathrm{biased,\;}N_{\mathrm{train}}='+str(no_training_data_per_feature)+'$')
#ax.plot(xx, yy_u, 'o', color=cmap(1.0), markersize=7, 
#        label=r'$\mathrm{unbiased,\;}N_{\mathrm{train}}='+str(no_training_data_per_feature)+'$')

ax.legend(fontsize=22)
ax.set_xlabel(r'$M$', fontsize=fs)
ax.set_ylabel(r'$\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}$', fontsize=fs)
#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_ylim([1.0e-12, 1.0e03])

Loops over several Ntrain values (`no_train_vec`), loading the corresponding aggregated biased/unbiased extracted results and averaging `Delta_{f-c}MSE_min` over model draws for each Nparams value, to build the full Nparams-vs-Ntrain scaling trend of biased vs. unbiased training performance.

In [ ]:
no_train_vec = [10, 15, 20, 30, 40, 60]

dict_allNtrain = dict()
for no_train in no_train_vec:
    name_end_extr = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
                     '_DimBasisParam' + name_dim_basis_param + '_NtrainPerFeat' + str(no_train) + 
                     '_batch' + batch_size_name + '_lr' + learning_rate_name + '_cutoff' + cutoff_name + 
                     '_decayexp' + decay_exp_name)
        
    filename = 'extracted_results_FULLbiased' + name_end_extr + '.pkl'
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_fullbiased_all = pickle.load(f)
    
    filename = 'extracted_results_UNbiased' + name_end_extr + '.pkl'
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_unbiased_all = pickle.load(f)
    
    dMSE_biased = []
    dMSE_unbiased = []
    for no_params in no_params_vec:
        dict_nopars_biased = dict_fullbiased_all[no_params]
        deltaMSE_biased = np.asarray(dict_nopars_biased['delta_minMSE_mean'])
        dMSE_biased.append(np.mean(deltaMSE_biased))
        dict_nopars_unbiased = dict_unbiased_all[no_params]
        deltaMSE_unbiased = np.asarray(dict_nopars_unbiased['delta_minMSE_mean'])
        dMSE_unbiased.append(np.mean(deltaMSE_unbiased))
    yy_b = np.asarray(dMSE_biased)
    yy_u = np.asarray(dMSE_unbiased)
    dict_Ntrain = dict()
    dict_Ntrain['deltaMSE_biased'] = yy_b
    dict_Ntrain['deltaMSE_unbiased'] = yy_u
    dict_allNtrain[no_train] = dict_Ntrain

Plots `Delta_{f-c}MSE_min` (mean over model draws) for the biased data-generating case against Nparams (M), with points colored by Ntrain via a colorbar, reproducing the paper's Nparams-scaling comparison of training performance between the full and cutoff models at fixed Dloc=3.

In [ ]:
fs = 28
figsize = (8,4)

cmap = matplotlib.colormaps['cividis']

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

cmap = matplotlib.colormaps['viridis']
xx = []
yy_b = []
yy_u = []
zz = []
for no_train in no_train_vec:
    dict_Ntrain = dict_allNtrain[no_train]
    y_b = dict_Ntrain['deltaMSE_biased']
    y_u = dict_Ntrain['deltaMSE_unbiased']
    for i in range(len(y_b)):
        xx.append(no_params_vec[i])
        yy_b.append(y_b[i])
        yy_u.append(y_u[i])
        zz.append(no_train)
xx = np.asarray(xx)
yy_b = np.asarray(yy_b)
yy_u = np.asarray(yy_u)
zz = np.asarray(zz)

fA = ax.scatter(xx, yy_b, marker='o', s=70, c=zz, cmap=cmap)
#fA = ax.scatter(xx, yy_u, marker='o', s=70, c=zz, cmap=cmap)
cbar = fig.colorbar(fA, ax=ax, extend='both')
cbar.ax.set_ylabel(r'$n_{\mathrm{train}}$', fontsize=fs)

#ax.legend(fontsize=22)
ax.set_xlabel(r'$M$', fontsize=fs)
ax.set_ylabel(r'$\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}$', fontsize=fs)
#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_ylim([1.0e-12, 1.0e03])

Saves the resulting figure(s) to disk in PNG and PDF format.

In [ ]:
#fig.savefig('DeltaMSEmin_unbiased_vs_M_Dloc3_chi50_L13_R3.png', bbox_inches='tight')
#fig.savefig('DeltaMSEmin_unbiased_vs_M_Dloc3_chi50_L13_R3.pdf', bbox_inches='tight')
fig.savefig('DeltaMSEmin_fullbiased_vs_M_Dloc3_chi50_L13_R3.png', bbox_inches='tight')
fig.savefig('DeltaMSEmin_fullbiased_vs_M_Dloc3_chi50_L13_R3.pdf', bbox_inches='tight')